### Download Datasaet from roboflow

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="LxSDqMlGhP5jKsgsCom8")
project = rf.workspace("khaleds-workspace-swezy").project("spot_classificatino2")
version = project.version(1)
dataset = version.download("folder")


loading Roboflow workspace...
loading Roboflow project...


### Walk Throught Dataset

In [2]:
import  os
from pathlib import Path
import numpy as np
import cv2
def walk_throught(dataset_path):
  for dir_path, dir_name, file_name in os.walk(dataset_path):
    print(f'Threr are {len(dir_name)} dirs and {len(file_name)} files in {dir_path} ')
walk_throught("/content/spot_classificatino2-1")

Threr are 2 dirs and 2 files in /content/spot_classificatino2-1 
Threr are 2 dirs and 0 files in /content/spot_classificatino2-1/valid 
Threr are 0 dirs and 299 files in /content/spot_classificatino2-1/valid/empty 
Threr are 0 dirs and 310 files in /content/spot_classificatino2-1/valid/not_empty 
Threr are 2 dirs and 0 files in /content/spot_classificatino2-1/train 
Threr are 0 dirs and 2746 files in /content/spot_classificatino2-1/train/empty 
Threr are 0 dirs and 2735 files in /content/spot_classificatino2-1/train/not_empty 


### Load Training Dataset

In [19]:
train_dir = Path("/content/spot_classificatino2-1/train")
test_dir = Path("/content/spot_classificatino2-1/valid")
X_train = []
y_train = []

for class_name, label in [("not_empty", 0), ("empty", 1)]:
  class_dir = train_dir/class_name
  train_images = list(class_dir.glob("*.jpg"))
  for img_path in train_images:
    img = cv2.imread(str(img_path))
    if img is not None:
      img = cv2.resize(img, (15, 15))
      X_train.append(img)
      y_train.append(label)


X_train = np.array(X_train).reshape(len(X_train), -1)
y_train = np.array(y_train)
print(f"train features shape: {X_train.shape}")
print(f"train labels shape: {y_train.shape}")

train features shape: (5481, 675)
train labels shape: (5481,)


### Load Test Dataset

In [18]:
X_test = []
y_test = []

for class_name, label in [("not_empty", 0), ("empty", 1)]:
  class_dir = test_dir/ class_name
  test_images = list(class_dir.glob("*.jpg"))
  for img_path in test_images:
    img = cv2.imread(str(img_path))
    if img is not None:
      img = cv2.resize(img, (15, 15))
      X_test.append(img)
      y_test.append(label)

X_test = np.array(X_test).reshape(len(X_test), -1)
y_test = np.array(y_test)
print(f"Test features shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")

Test features shape: (609, 675)
Test labels shape: (609,)


### Scale Train and Test features

In [21]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Scaled train features shape: {X_train_scaled.shape}")
print(f"Scaled test features shape: {X_test_scaled.shape}")

Scaled train features shape: (5481, 675)
Scaled test features shape: (609, 675)


### Train SVC model

In [6]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, f1_score, average_precision_score
svc_model = SVC(kernel='rbf')
param_grid = {
    'C': [0.1, 1, 10, 50, 100],
    'gamma': ['scale', 0.0001, 0.001, 0.01, 0.1, 1, 10],
}
grid_search = GridSearchCV(estimator= svc_model,
                           param_grid=param_grid,
                           cv=5,
                           scoring= 'accuracy')

In [7]:
grid_search.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10, 50, 100],
                         'gamma': ['scale', 0.0001, 0.001, 0.01, 0.1, 1, 10]},
             scoring='accuracy')

### Save Model and Scaler

In [16]:
import pickle
with open('svc_model.pkl', 'wb') as f:
  pickle.dump(grid_search.best_estimator_, f)
with open('scaler.pkl', 'wb') as f:
  pickle.dump(scaler,f)

### Evaluate Model

In [13]:
model = grid_search.best_estimator_
y_predict= model.predict(X_test_scaled)
print(classification_report(y_test, y_predict))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       310
           1       1.00      1.00      1.00       299

    accuracy                           1.00       609
   macro avg       1.00      1.00      1.00       609
weighted avg       1.00      1.00      1.00       609



In [14]:
print(f"AP = {average_precision_score(y_test, y_predict):.2f}")
print(f"Accuracy = {accuracy_score(y_test, y_predict):.2f}")

AP = 1.00
Accuracy = 1.00
